In [2]:
import os

from google import genai
from google.colab import userdata, files

from sentence_transformers import SentenceTransformer
import chromadb

from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
api_key = userdata.get("GeminiAPIKey3")

if not api_key:
    raise ValueError(
        "GEMINI_API_KEY not found. Add it to Colab Secrets."
    )

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [8]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Say Hello in one sentence."
)

print(response.text)

Hello, I hope you are having a wonderful day!


In [11]:
print("Please upload one or more PDF files:")
uploaded = files.upload()

pdf_texts = []

for filename in uploaded.keys():
    if filename.lower().endswith(".pdf"):
        reader = PdfReader(filename)
        text = ""

        for page_num, page in enumerate(reader.pages):
            page_text = page.extract_text()

            if page_text:
                text += f"\n--- Page {page_num + 1} ---\n" + page_text

        pdf_texts.append(text)

        print(f"Loaded '{filename}' ({len(reader.pages)} pages).")

if not pdf_texts:
    raise ValueError(
        "No valid PDF files uploaded. Please re-run and upload a .pdf file."
    )

full_pdf_content = "\n\n".join(pdf_texts)

print(f"\nTotal extracted text: {len(full_pdf_content):,} characters")

Please upload one or more PDF files:


Saving GEN AI_LLM Chatbot_97.pdf to GEN AI_LLM Chatbot_97.pdf
Loaded 'GEN AI_LLM Chatbot_97.pdf' (3 pages).

Total extracted text: 1,834 characters


In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_text(full_pdf_content)

print(f"Extracted and split document into {len(chunks)} text chunks.")

Extracted and split document into 5 text chunks.


In [13]:
print("Loading embedding model and building vector index...")

embedder = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.Client()

# Reset collection for clean execution
try:
    chroma_client.delete_collection(name="pdf_rag_collection")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="pdf_rag_collection"
)

# Embed chunks
chunk_embeddings = embedder.encode(chunks).tolist()

# Create unique IDs
chunk_ids = [f"doc_chunk_{i}" for i in range(len(chunks))]

# Add documents and embeddings to ChromaDB
collection.add(
    documents=chunks,
    embeddings=chunk_embeddings,
    ids=chunk_ids
)

print("PDF Vector Indexing Complete!\n")
print(f"Total chunks indexed: {len(chunks)}")

Loading embedding model and building vector index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

PDF Vector Indexing Complete!

Total chunks indexed: 5


In [14]:
def retrieve_pdf_context(query: str, top_k: int = 3) -> list[str]:
    query_embedding = embedder.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    return results["documents"][0]


def ask_pdf(query: str):
    context_passages = retrieve_pdf_context(query, top_k=3)

    context_str = "\n".join(
        f"- {p}" for p in context_passages
    )

    prompt = f"""
You are an intelligent document analysis assistant.
Answer the question using ONLY the provided PDF context below.

If the information is not contained within the provided context,
state clearly:

"I cannot find the answer in the provided PDF."

PDF CONTEXT:
{context_str}

Question:
{query}

Answer:
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text, context_passages

In [ ]:
# ============================================================
# PDF CHATBOT
# ============================================================

def retrieve_pdf_context(query: str, top_k: int = 3):
    # Convert the user's question into an embedding
    query_embedding = embedder.encode([query]).tolist()

    # Search ChromaDB for the most relevant PDF chunks
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    # Return the retrieved documents
    return results["documents"][0]


def ask_pdf(query: str):
    # Retrieve relevant passages from the PDF
    context_passages = retrieve_pdf_context(query, top_k=3)

    # Combine the retrieved passages
    context_str = "\n".join(
        f"- {p}" for p in context_passages
    )

    # Create the prompt
    prompt = f"""
You are an intelligent PDF document analysis assistant.

Answer the question using ONLY the PDF context provided below.

Do not use outside knowledge.

If the answer is not available in the provided PDF context,
say exactly:

"I cannot find the answer in the provided PDF."

PDF CONTEXT:
{context_str}

QUESTION:
{query}

ANSWER:
"""

    # Generate answer using Gemini
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text, context_passages


# ============================================================
# START PDF CHATBOT
# ============================================================

print("=" * 60)
print("PDF CHATBOT READY!")
print("Type your question below.")
print("Type 'exit' to quit.")
print("=" * 60)

while True:

    user_query = input("\nAsk a question about your PDF: ")

    # Check for exit command
    if user_query.strip().lower() in ["exit", "quit", "q"]:
        print("\nExiting PDF Chatbot. Goodbye!")
        break

    # Ignore empty input
    if not user_query.strip():
        continue

    try:

        # Get answer from PDF
        answer, context = ask_pdf(user_query)

        # Show retrieved PDF snippets
        print("\n--- RETRIEVED PDF SNIPPETS ---")

        for i, snippet in enumerate(context, start=1):
            print(f"\n[{i}] {snippet[:300]}...")

        # Show Gemini answer
        print("\n--- GEMINI RESPONSE ---")
        print(answer)

        print("\n" + "-" * 60)

    except Exception as e:

        print("\n❌ ERROR:")
        print(e)
        print("-" * 60)

PDF CHATBOT READY!
Type your question below.
Type 'exit' to quit.

Ask a question about your PDF: What is the pdf all about ?

--- RETRIEVED PDF SNIPPETS ---

[1] --- Page 1 ---
Name:- Pranjal Mule Roll no:-97 
 
NCRD’s Sterling Institute of Management Studies 
(NAAC Accredited A+ Grade) 
 
Nerul, Navi Mumbai 
 
2025-2027 
GENERATIVE AI 10-Day Workshop 
Project Documentation 
On 
Basic LLM Chatbot Using Gemini 
Under the Guidance 
of 
Mr. NAVNEET SIR 
By 
Nam...

[2] --- Page 3 ---
Name:- Pranjal Mule Roll no:-97 
 
Step 4 : Initialize the Gemini Client 
 
Step 5 : Define the Prompt 
 
Step 6 : Generate the Response 
 
Step 7 : Display the Response 
 
Conclusion:- 
The assignment was successfully completed by creating a basic LLM application using 
Google Gemini...

[3] --- Page 2 ---
Name:- Pranjal Mule Roll no:-97 
1. Title :- 
Basic LLM Chatbot Using Google Gemini API 
2. Aim :- 
To create a simple Large Language Model (LLM) application using the Google Gemini API 
in Google Colab a